# VeritasAI · Approach A · Frame-level EfficientNet-B0 baseline
Each face frame is classified independently; the video score is the mean frame probability.
Mirrors survey papers [2] and [6]. GPU: T4, ~20-40 min.

In [ ]:
# ==== 0. SETUP (same in every notebook) ===================================
# Kaggle: Settings -> Accelerator = GPU T4 x2 (or None for notebook 00), Internet = ON
# Colab : Runtime -> Change runtime type -> T4 GPU
import os, sys
KAGGLE = os.path.exists("/kaggle")
WORK = "/kaggle/working" if KAGGLE else "/content"
REPO_URL = "https://github.com/Bhoomified/VeritasAI.git"   # your repo (push the new ml/ files first!)
os.chdir(WORK)
if not os.path.exists("VeritasAI"):
    os.system(f"git clone -q {REPO_URL}")
os.chdir(f"{WORK}/VeritasAI"); os.system("git pull -q")
os.system('pip -q install "opencv-python-headless==4.10.0.84" onnx onnxruntime tabulate')
if not KAGGLE:   # Colab: keep outputs on Google Drive so a disconnect never loses work
    from google.colab import drive; drive.mount("/content/drive")
STORE = "/kaggle/working" if KAGGLE else "/content/drive/MyDrive/veritas"
os.makedirs(STORE, exist_ok=True)
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")

In [ ]:
# ==== where the face strips from notebook 00 live ==========================
# Kaggle: "Add Input" -> your notebook-00 output; it appears under /kaggle/input/<name>/
FACES = "/kaggle/input/veritas-faces/faces" if KAGGLE else f"{STORE}/faces"
TRAIN_SET, CROSS_SET = "ffpp", "celebdf"          # train on FF++, test unseen on Celeb-DF
F = f"{FACES}/{TRAIN_SET}"
TR, VA, TE = f"{F}/{TRAIN_SET}_train.csv", f"{F}/{TRAIN_SET}_val.csv", f"{F}/{TRAIN_SET}_test.csv"
CROSS = f"{FACES}/{CROSS_SET}/{CROSS_SET}_test.csv"
RESULTS = f"{STORE}/results.csv"; CK = f"{STORE}/checkpoints"

In [ ]:
!python -m ml.finetune --mode frame --backbone efficientnet_b0 --unfreeze 4 --epochs 8 --bs 16 \
   --faces {F} --train {TR} --val {VA} --test {TE} --cross {CROSS} --out {CK} --results {RESULTS} --workers 4

In [ ]:
# lighter variant for the efficiency comparison
!python -m ml.finetune --mode frame --backbone mobilenet_v3_large --unfreeze 4 --epochs 8 --bs 16 \
   --faces {F} --train {TR} --val {VA} --test {TE} --cross {CROSS} --out {CK} --results {RESULTS} --workers 4
import pandas as pd; pd.read_csv(RESULTS)